# 1- Prepare Notebook Environment

- Create a Jupyter Notebook: government_offices_data_transformation.ipynb inside /scripts.
- Ensure all required Python libraries are installed and imported: pandas, geopandas, shapely, osmnx, requests, json.

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import requests
import json

# 2- 📊 Discover and Fetch data

This script uses the OSMnx library to fetch data on government and administrative offices in Berlin, Germany. It iteratively queries OpenStreetMap using specific tags (e.g., office=government, amenity=townhall). The results are then combined, deduplicated using the unique OSM ID (osmid), and finally filtered to isolate entities with German-specific names (like 'Bürgeramt' or 'Finanzamt') for a targeted administrative dataset.

In [2]:
# Tags for different types of government offices
tags_list = [
    {"office": "government"},
    {"office": "administrative"},
    {"amenity": "townhall"},
    {"amenity": "public_building"},
    {"office": "employment_agency"},
]

# A list to store the data we fetch
all_offices = []

# Step 1: Fetch data for each tag
for tags in tags_list:
    try:
        gdf = ox.features_from_place("Berlin, Germany", tags)
        all_offices.append(gdf)
    except Exception:
        pass  # If no data found for a tag, just skip it

# Step 2: Combine all data into one dataframe
if all_offices:
    government_offices_gdf = pd.concat(all_offices, ignore_index=False)

    # Reset index so 'osmid' (if it exists) becomes a column
    government_offices_gdf = government_offices_gdf.reset_index()

    # Step 3: Remove duplicate entries
    if 'osmid' in government_offices_gdf.columns:
        government_offices_gdf = government_offices_gdf.drop_duplicates(subset=['osmid'])
    elif {'element_type', 'osmid'}.issubset(government_offices_gdf.columns):
        government_offices_gdf = government_offices_gdf.drop_duplicates(subset=['element_type', 'osmid'])
    else:
        government_offices_gdf = government_offices_gdf.drop_duplicates()

    # Step 4: Optionally filter offices with German government-related names
    german_keywords = [
        'Bürgeramt', 'Bezirksamt', 'Finanzamt', 'Standesamt',
        'Sozialamt', 'Jobcenter', 'Ausländerbehörde',
        'Landesamt', 'Senatsverwaltung', 'Ordnungsamt'
    ]

    if 'name' in government_offices_gdf.columns:
        german_offices = government_offices_gdf[
            government_offices_gdf['name'].str.contains('|'.join(german_keywords), case=False, na=False)
        ]
    else:
        german_offices = pd.DataFrame()  # Empty if no 'name' column found

    # Step 5: Show how many results were found
    print(f"Total unique government office entries: {len(government_offices_gdf)}")
    print(f"Total unique German government office entries: {len(german_offices)}")

else:
    print("No government office data found.")


Total unique government office entries: 441
Total unique German government office entries: 123


# 3- 📋 Initial Data overview and Inspection

This snippet displays the basic structure of the fetched dataset, showing it contains 441 unique government office entries with 177 columns. The output confirms the presence of essential location data (elementid, geometry) and initial address fields, while also revealing that the dataset is highly sparse (many NaN values) due to the vast number of specific OpenStreetMap tags captured.

In [3]:
# Display basic dataset information
if government_offices_gdf is not None:
    print(f"Dataset Shape: {government_offices_gdf.shape}")
    print(f"Total Columns: {len(government_offices_gdf.columns)}")
    display(government_offices_gdf.head())


Dataset Shape: (441, 177)
Total Columns: 177


,element,id,geometry,addr:city,addr:housenumber,addr:postcode,addr:street,government,level,name,...,historic_name:de,historic_name:en,building:parts,surveillance,name:prefix,source:website,ele,brand,brand:wikipedia,internet_access:ssid
0,node,331398399,POINT (13.34756 52.42794),Berlin,87,12249,Gallwitzallee,public_service,1,Bürgeramt Lankwitz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,356925920,POINT (13.32987 52.50927),Berlin,87,10623,Fasanenstraße,NaN,NaN,Bundesanstalt für Immobilienaufgaben,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,361001240,POINT (13.26794 52.50944),Berlin,12-14,14052,Heerstraße,public_service,NaN,Bürgeramt Heerstraße,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,371520086,POINT (13.18393 52.51935),Berlin,25-30,13593,Wilhelmstraße,NaN,NaN,Bundesanstalt für Geowissenschaften und Rohsto...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,434480600,POINT (13.20134 52.53527),Berlin,1,13597,Am Wall,register_office,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# save government_offices_gdf to excel file
#government_offices_gdf.to_excel('government_offices_berlin.xlsx', index=False)


In [5]:
#expland all columns to see more details
#pd.set_option('display.max_columns', None)
#print(government_offices_gdf.head(5))

# 4- 🧹 Data Consolidation and merging of duplicate columns

A sequential merging strategy is done to consolidate similar information from highly sparse OpenStreetMap columns into a single, preferred column. The fill_column_sequential function is applied to five key attributes:

- name: Missing values are filled using German, English, and official names.

- opening_hours: Missing hours are sourced from opening_hours:signed.

- website: Missing websites are sourced from contact:website.

- contact:phone: Missing phone numbers are sourced from the generic phone column.

- email: Missing emails are sourced from contact:email.

This process significantly reduces the number of null values in crucial columns, preparing the data for the next enrichment step using official Berlin municipal sources.

In [6]:
def fill_column_sequential(df, main_column, fallback_columns):
    """
    Fill missing values in main column by checking fallback columns sequentially.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to modify
    main_column : str
        The main column name to fill
    fallback_columns : list
        List of column names to check in sequence for filling missing values
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with the main column filled
    
    Usage example:
    --------------
    df = fill_column_sequential(df, 'main_col', ['fallback_col1', 'fallback_col2'])
    
    government_offices_gdf(
    'name', 
    ['name:de', 'name:en', 'official_name'])
    """
    # Start with the main column
    result = df[main_column].copy()
    
    # Check each fallback column in sequence
    for col in fallback_columns:
        if col in df.columns:
            result = result.fillna(df[col])
        else:
            print(f"Warning: Column '{col}' not found in dataframe, skipping...")
    
    # Assign back to dataframe
    df[main_column] = result
    
    # Print summary
    remaining_nulls = df[main_column].isna().sum()
    print(f"Column '{main_column}' filled successfully")
    print(f"Remaining null values: {remaining_nulls}")
    
    return df

In [7]:
government_offices_gdf['name'].isna().sum()

np.int64(18)

In [8]:
government_offices_gdf= fill_column_sequential(df=government_offices_gdf, 
                       main_column='name', 
                       fallback_columns=['name:de', 'name:en', 'official_name'])


Column 'name' filled successfully
Remaining null values: 18


In [9]:
print(government_offices_gdf['opening_hours'].isna().sum())
government_offices_gdf= fill_column_sequential(df=government_offices_gdf, 
                       main_column='opening_hours', 
                       fallback_columns=['opening_hours:signed'])

307
Column 'opening_hours' filled successfully
Remaining null values: 273


In [10]:
print(government_offices_gdf['website'].isna().sum())
government_offices_gdf= fill_column_sequential(df=government_offices_gdf, 
                       main_column='website', 
                       fallback_columns=['contact:website'])

271
Column 'website' filled successfully
Remaining null values: 177


In [11]:
print(government_offices_gdf['contact:phone'].isna().sum())
government_offices_gdf= fill_column_sequential(df=government_offices_gdf, 
                       main_column='contact:phone', 
                       fallback_columns=['phone'])

386
Column 'contact:phone' filled successfully
Remaining null values: 296


In [12]:
print(government_offices_gdf['email'].isna().sum())
government_offices_gdf= fill_column_sequential(df=government_offices_gdf, 
                       main_column='email', 
                       fallback_columns=['contact:email'])

399
Column 'email' filled successfully
Remaining null values: 372


## 5- 🏗️ Standardizing the column names and the Dataset Structure

The crucial step of standardizing and preparing the OSMnx data brings into a clean, targeted structure suitable for database entry and the upcoming enrichment steps.

- Column Mapping: It Normalize all columns to snake_case and lowercase and by renaming the highly specific OSM column names (e.g., addr:housenumber, contact:phone) to the desired, consistent schema names (e.g., housenumber, phone_number).

- Schema Enforcement: It enforces a predefined, 17-column structure (desired_columns).

- Data Alignment: It selects only the mapped columns and adds any missing required columns (like coordinate_type) with None placeholders to ensure the final GeoDataFrame is structurally complete, regardless of the availability of data from the OSM source.

The resulting GeoDataFrame now has the correct shape and column names, making it ready to be compared and merged with the highly accurate Berlin Open Data WFS you identified earlier.

In [13]:

# Column mapping for government offices data
column_mapping = {
    # office id and name related columns
    'id': 'office_id',
    'name': 'office_name',
    'government': 'office_type',

    # Address related columns
    'addr:housenumber': 'housenumber',
    'addr:street': 'street',
    'addr:postcode': 'postal_code',
    'addr:city': 'city',

    # Contact information 
    'contact:phone': 'phone_number',
    'email': 'email',
    'website': 'website',
    'opening_hours': 'opening_hours',
    'wheelchair': 'wheelchair_accessible',
    
    # Location (will be calculated from geometry)
    'latitude': 'latitude',
    'longitude': 'longitude',
    'geometry': 'geometry',

}

# Apply the mapping
government_offices_mapped = government_offices_gdf.rename(columns=column_mapping)

# Select and reorder columns to match the desired 19-column structure
desired_columns = [
    'office_id',
    'office_name', 
    'office_type',
    'street',
    'housenumber',
    'postal_code',
    'city',
    'phone_number',
    'email',
    'website',
    'opening_hours',
    'wheelchair_accessible',
    'latitude',
    'longitude',
    'geometry',
    'coordinate_type'
]

# Create the final dataframe with only available columns
available_columns = [col for col in desired_columns if col in government_offices_mapped.columns]
government_offices_final = government_offices_mapped[available_columns].copy()

# Add missing columns with None/NaN values
for col in desired_columns:
    if col not in government_offices_final.columns:
        government_offices_final[col] = None

# Reorder to match desired structure
government_offices_final = government_offices_final[desired_columns]

print(f"Final structure: {government_offices_final.shape}")
print(f"Columns: {government_offices_final.columns.tolist()}")
government_offices_final.head()

Final structure: (441, 16)
Columns: ['office_id', 'office_name', 'office_type', 'street', 'housenumber', 'postal_code', 'city', 'phone_number', 'email', 'website', 'opening_hours', 'wheelchair_accessible', 'latitude', 'longitude', 'geometry', 'coordinate_type']


,office_id,office_name,office_type,street,housenumber,postal_code,city,phone_number,email,website,opening_hours,wheelchair_accessible,latitude,longitude,geometry,coordinate_type
0,331398399,Bürgeramt Lankwitz,public_service,Gallwitzallee,87,12249,Berlin,NaN,NaN,https://service.berlin.de/standort/122276/,Mo 08:00-15:00; Tu 10:00-18:00; We 07:30-14:30...,no,None,None,POINT (13.34756 52.42794),None
1,356925920,Bundesanstalt für Immobilienaufgaben,NaN,Fasanenstraße,87,10623,Berlin,NaN,NaN,NaN,NaN,yes,None,None,POINT (13.32987 52.50927),None
2,361001240,Bürgeramt Heerstraße,public_service,Heerstraße,12-14,14052,Berlin,+49 30 902917777,NaN,NaN,"Mo 09:00-13:30,14:30-18:00; Tu 09:00-18:00; We...",yes,None,None,POINT (13.26794 52.50944),None
3,371520086,Bundesanstalt für Geowissenschaften und Rohsto...,NaN,Wilhelmstraße,25-30,13593,Berlin,NaN,NaN,http://www.bgr.bund.de/,NaN,NaN,None,None,POINT (13.18393 52.51935),None
4,434480600,NaN,register_office,Am Wall,1,13597,Berlin,NaN,NaN,NaN,NaN,NaN,None,None,POINT (13.20134 52.53527),None


## 6- 🗺️ Geo-Enrichment with Official Berlin District Data

The enrichment of the government office datais done by linking each office's geospatial coordinates to the official Berlin administrative boundaries (districts and neighborhoods).

- District Data Preparation: It loads a separate GeoJSON file containing the authoritative boundaries (lor_ortsteile.geojson). It creates a standardized district_id column using a provided mapping of district names to official ID strings.

- Spatial Join: A spatial join (gpd.sjoin) is executed, using the within predicate to check which district/neighborhood polygon each office's geometry point falls inside.

- Column Finalization: The resulting merged columns are renamed to match the final schema (district, neighborhood, and neighborhood_id), and redundant columns are dropped.

This process successfully appends official and accurate district, neighborhood, district_id, and neighborhood_id attributes to the OSMnx data, completing the administrative area information for all 441 records.

In [14]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson
berlin_districts_gdf = gpd.read_file("../sources/lor_ortsteile.geojson")
# rename BEZIRK to district, OTEIL to neighborhood
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

#using this mapping to create district_id column (string) in berlin_districts_gdf
berlin_districts_gdf['district_id'] = berlin_districts_gdf['BEZIRK'].map(district_mapping).astype(str)


print(berlin_districts_gdf.shape)
print(berlin_districts_gdf.columns)
print(berlin_districts_gdf.head())

(96, 9)
Index(['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL',
       'BEZIRK', 'FLAECHE_HA', 'geometry', 'district_id'],
      dtype='object')
             gml_id spatial_name spatial_alias spatial_type         OTEIL  \
0  re_ortsteil.0101         0101         Mitte      Polygon         Mitte   
1  re_ortsteil.0102         0102        Moabit      Polygon        Moabit   
2  re_ortsteil.0103         0103  Hansaviertel      Polygon  Hansaviertel   
3  re_ortsteil.0104         0104    Tiergarten      Polygon    Tiergarten   
4  re_ortsteil.0105         0105       Wedding      Polygon       Wedding   

  BEZIRK  FLAECHE_HA                                           geometry  \
0  Mitte   1063.8748  POLYGON ((13.41649 52.52696, 13.41635 52.52702...   
1  Mitte    768.7909  POLYGON ((13.33884 52.51974, 13.33884 52.51974...   
2  Mitte     52.5337  POLYGON ((13.34322 52.51557, 13.34323 52.51557...   
3  Mitte    516.0672  POLYGON ((13.36879 52.49878, 13.36891 52.49877...  

In [15]:
# Spatial join
government_offices_joined = gpd.sjoin(
    government_offices_final,
    berlin_districts_gdf[["BEZIRK", "OTEIL","geometry", "district_id", "spatial_name"]],
    how="left",
    predicate="within"
)

print(government_offices_joined.shape)
print(government_offices_joined.columns)
print(government_offices_joined.head())

(441, 21)
Index(['office_id', 'office_name', 'office_type', 'street', 'housenumber',
       'postal_code', 'city', 'phone_number', 'email', 'website',
       'opening_hours', 'wheelchair_accessible', 'latitude', 'longitude',
       'geometry', 'coordinate_type', 'index_right', 'BEZIRK', 'OTEIL',
       'district_id', 'spatial_name'],
      dtype='object')
   office_id                                        office_name  \
0  331398399                                 Bürgeramt Lankwitz   
1  356925920               Bundesanstalt für Immobilienaufgaben   
2  361001240                               Bürgeramt Heerstraße   
3  371520086  Bundesanstalt für Geowissenschaften und Rohsto...   
4  434480600                                                NaN   

       office_type         street housenumber postal_code    city  \
0   public_service  Gallwitzallee          87       12249  Berlin   
1              NaN  Fasanenstraße          87       10623  Berlin   
2   public_service     Heerstraß

In [16]:
# rename BEZIRK to district, OTEIL to neighborhood
government_offices_joined = government_offices_joined.rename(columns={'BEZIRK': 'district', 'OTEIL': 'neighborhood', 'spatial_name': 'neighborhood_id'})
drop_columns = ['index_right']
government_offices_joined = government_offices_joined.drop(columns=drop_columns)

In [17]:

print(government_offices_joined.shape)
print(government_offices_joined.columns)
print(government_offices_joined.head())

(441, 20)
Index(['office_id', 'office_name', 'office_type', 'street', 'housenumber',
       'postal_code', 'city', 'phone_number', 'email', 'website',
       'opening_hours', 'wheelchair_accessible', 'latitude', 'longitude',
       'geometry', 'coordinate_type', 'district', 'neighborhood',
       'district_id', 'neighborhood_id'],
      dtype='object')
   office_id                                        office_name  \
0  331398399                                 Bürgeramt Lankwitz   
1  356925920               Bundesanstalt für Immobilienaufgaben   
2  361001240                               Bürgeramt Heerstraße   
3  371520086  Bundesanstalt für Geowissenschaften und Rohsto...   
4  434480600                                                NaN   

       office_type         street housenumber postal_code    city  \
0   public_service  Gallwitzallee          87       12249  Berlin   
1              NaN  Fasanenstraße          87       10623  Berlin   
2   public_service     Heerstraße  

In [18]:

print(government_offices_joined['district_id'].unique())


['11006006' '11004004' '11005005' '11011011' '11002002' '11001001'
 '11010010' '11007007' '11008008' '11003003' '11012012' '11009009' nan]


In [19]:
# drop  where district_id is null
government_offices_joined = government_offices_joined[government_offices_joined['district_id'].notnull()]


## 7- 🗺️ Coordinate Extraction
In this step two crucial task are performed such as cleaning steps on the geospatial dataset to ensure data quality and integrity before enrichment begins:

- Spatial Filtering (Out-of-Bounds Data): It removes the few records where the preceding spatial join failed to assign a district_id (indicated by nan in the unique list). This drop action removes any offices that may have been outside Berlin's official administrative boundaries, resulting in a cleaner dataset of 437 records.

- Coordinate Finalization: It extracts the precise longitude (x) and latitude (y) values directly from the geometry Point object and writes them into their respective columns (longitude, latitude). Concurrently, it sets the coordinate_type to 'point', formally documenting the source of the geographic coordinates.

The dataset is now completely cleaned, structured, spatially enriched, and ready for verification and attribute filling using the authoritative Berlin Open Data source.

In [20]:
# Function to safely extract coordinates
def get_coords(geom, coord):
    if isinstance(geom, Point):
        return geom.x if coord == 'x' else geom.y
    else:
        return None

# Extract longitude and latitude
office_gdf = government_offices_joined  # your dataframe

office_gdf['longitude'] = office_gdf.geometry.apply(lambda g: get_coords(g, 'x'))
office_gdf['latitude'] = office_gdf.geometry.apply(lambda g: get_coords(g, 'y'))

# Set coordinate_type to 'point' only if geometry is Point, else None
office_gdf['coordinate_type'] = office_gdf.geometry.apply(lambda g: 'point' if isinstance(g, Point) else None)


In [21]:
print(office_gdf.shape)
print(office_gdf.columns)
print(office_gdf.head())

(437, 20)
Index(['office_id', 'office_name', 'office_type', 'street', 'housenumber',
       'postal_code', 'city', 'phone_number', 'email', 'website',
       'opening_hours', 'wheelchair_accessible', 'latitude', 'longitude',
       'geometry', 'coordinate_type', 'district', 'neighborhood',
       'district_id', 'neighborhood_id'],
      dtype='object')
   office_id                                        office_name  \
0  331398399                                 Bürgeramt Lankwitz   
1  356925920               Bundesanstalt für Immobilienaufgaben   
2  361001240                               Bürgeramt Heerstraße   
3  371520086  Bundesanstalt für Geowissenschaften und Rohsto...   
4  434480600                                                NaN   

       office_type         street housenumber postal_code    city  \
0   public_service  Gallwitzallee          87       12249  Berlin   
1              NaN  Fasanenstraße          87       10623  Berlin   
2   public_service     Heerstraße  

In [22]:
# Analyze how many values are missing in each column

if office_gdf is not None:
    missing_analysis = pd.DataFrame({
        'Column': office_gdf.columns,
        'Non-Null Count': office_gdf.count(),
        'Null Count': office_gdf.isnull().sum(),
        'Null Percentage': (office_gdf.isnull().sum() / len(office_gdf) * 100).round(2)
    }).sort_values('Null Percentage')

    display(missing_analysis)

    high_missing = missing_analysis[missing_analysis['Null Percentage'] > 85]
    print(f"Columns with >85% missing values: {len(high_missing)}")
    if len(high_missing) > 0:
        for col in high_missing['Column']:
            print(f" - {col}")

,Column,Non-Null Count,Null Count,Null Percentage
office_id,office_id,437,0,0.00
neighborhood,neighborhood,437,0,0.00
district,district,437,0,0.00
geometry,geometry,437,0,0.00
district_id,district_id,437,0,0.00
neighborhood_id,neighborhood_id,437,0,0.00
office_name,office_name,420,17,3.89
housenumber,housenumber,326,111,25.40
street,street,323,114,26.09
postal_code,postal_code,317,120,27.46


Columns with >85% missing values: 0


## 8- 🌐 Validating and Cleaning Geospatial Data

This final cleaning step ensures the geometric integrity and spatial uniqueness of the office locations, resulting in the final, production-ready version of the dataset ready for attribute enrichment.

1.  **CRS Validation:** The data's **Coordinate Reference System (CRS)** is confirmed to be the standard global web system, **EPSG:4326 (WGS84)**, requiring no conversion.

2.  **Geometry Integrity:** The script checks for and finds **no invalid geometries** and fixes a single **multipart geometry** (e.g., a multi-point feature). This item is successfully **exploded** into a single row, slightly increasing the row count from 437 to 438, ensuring each location is represented by a simple point.

3.  **Deduplication:** A rigorous two-stage deduplication process is applied:
    * **Exact Duplicates:** Records with the same **`office_id`** are removed, restoring the row count to 437.
    * **Near Duplicates:** A custom function searches for offices with the **same name** located within approximately **10 meters** of each other. This step removes one final overlapping or redundant record, resulting in the final clean count of **436 unique offices**.

The resulting **436 unique government office entries** are geometrically sound, spatially unique, and contain all the enriched administrative boundary information.

In [23]:
# Step 1: Check current CRS (Coordinate Reference System)
print(f"Current CRS: {office_gdf.crs}")

Current CRS: epsg:4326


In [24]:
# Step 2: Convert to EPSG:4326 (WGS84) if needed
if office_gdf.crs != 'EPSG:4326':
    office_gdf = office_gdf.to_crs('EPSG:4326')
    print("CRS converted to EPSG:4326")
else:
    print("CRS is already EPSG:4326")

CRS is already EPSG:4326


In [25]:
# Step 3: Check for invalid geometries
invalid_geometries = ~office_gdf.geometry.is_valid
invalid_count = invalid_geometries.sum()

print(f"Invalid geometries found: {invalid_count}")

# Fix invalid geometries if any exist
if invalid_count > 0:
    office_gdf.loc[invalid_geometries, 'geometry'] = (
        office_gdf.loc[invalid_geometries, 'geometry'].buffer(0)
    )
    print("Invalid geometries fixed")

Invalid geometries found: 0


In [26]:
# Step 4: Check for multipart geometries
multipart_mask = office_gdf.geometry.type.isin(['MultiPoint', 'MultiPolygon'])
multipart_count = multipart_mask.sum()

print(f"Multipart geometries found: {multipart_count}")

Multipart geometries found: 1


In [27]:
# Step 5: Explode multipart geometries to single parts
if multipart_count > 0:
    office_gdf = office_gdf.explode(index_parts=False)
    office_gdf = office_gdf.reset_index(drop=True)
    print(f"Multipart geometries exploded. New row count: {len(office_gdf)}")

Multipart geometries exploded. New row count: 438


In [28]:
# Step 7: Remove duplicates based on name and location proximity
# First, check for exact name duplicates
print(f"Rows before duplicate removal: {len(office_gdf)}")

# Remove exact duplicates based on office_id
office_gdf = office_gdf.drop_duplicates(subset=['office_id'], keep='first')

print(f"Rows after removing exact duplicates: {len(office_gdf)}")

Rows before duplicate removal: 438
Rows after removing exact duplicates: 437


In [29]:
# Step 8: Check for near-duplicate locations (within ~10 meters)

# Create a function to find nearby duplicates
def remove_nearby_duplicates(gdf, distance_threshold=0.0001):  # ~10 meters in degrees
    """
    Remove duplicates that are very close to each other
    distance_threshold in degrees (0.0001 ≈ 10 meters)
    """
    to_drop = []
    
    for idx, row in gdf.iterrows():
        if idx in to_drop:
            continue
        
        # Find nearby points
        nearby = gdf[
            (gdf.index != idx) & 
            (abs(gdf['latitude'] - row['latitude']) < distance_threshold) &
            (abs(gdf['longitude'] - row['longitude']) < distance_threshold)
        ]
        
        # If same name and nearby, mark for removal
        if len(nearby) > 0 and 'office_name' in gdf.columns:
            same_name = nearby[nearby['office_name'] == row['office_name']]
            to_drop.extend(same_name.index.tolist())
    
    return gdf.drop(to_drop)

# Apply duplicate removal
office_gdf_cleaned = remove_nearby_duplicates(office_gdf)

print(f"Rows after removing nearby duplicates: {len(office_gdf_cleaned)}")

Rows after removing nearby duplicates: 436


In [30]:
# Step 9: Reset index after all cleaning
office_gdf_cleaned = office_gdf_cleaned.reset_index(drop=True)

print(f"Final cleaned dataset shape: {office_gdf_cleaned.shape}")
print(f"Final columns: {office_gdf_cleaned.columns.tolist()}")

Final cleaned dataset shape: (436, 20)
Final columns: ['office_id', 'office_name', 'office_type', 'street', 'housenumber', 'postal_code', 'city', 'phone_number', 'email', 'website', 'opening_hours', 'wheelchair_accessible', 'latitude', 'longitude', 'coordinate_type', 'district', 'neighborhood', 'district_id', 'neighborhood_id', 'geometry']


## 9- Summary of Data Cleaning

In [31]:
# Display summary of cleaned data
print("=" * 70)
print("DATA CLEANING SUMMARY")
print("=" * 70)
print(f"Total rows: {len(office_gdf_cleaned)}")
print(f"Total columns: {len(office_gdf_cleaned.columns)}")
print(f"\nColumn completeness:")

for col in office_gdf_cleaned.columns:
    if col != 'geometry':
        non_null = office_gdf_cleaned[col].notna().sum()
        percentage = (non_null / len(office_gdf_cleaned) * 100)
        print(f"  {col:30s}: {non_null:4d} ({percentage:5.1f}%)")

DATA CLEANING SUMMARY
Total rows: 436
Total columns: 20

Column completeness:
  office_id                     :  436 (100.0%)
  office_name                   :  419 ( 96.1%)
  office_type                   :  232 ( 53.2%)
  street                        :  322 ( 73.9%)
  housenumber                   :  325 ( 74.5%)
  postal_code                   :  316 ( 72.5%)
  city                          :  315 ( 72.2%)
  phone_number                  :  142 ( 32.6%)
  email                         :   67 ( 15.4%)
  website                       :  261 ( 59.9%)
  opening_hours                 :  165 ( 37.8%)
  wheelchair_accessible         :  193 ( 44.3%)
  latitude                      :  270 ( 61.9%)
  longitude                     :  270 ( 61.9%)
  coordinate_type               :  270 ( 61.9%)
  district                      :  436 (100.0%)
  neighborhood                  :  436 (100.0%)
  district_id                   :  436 (100.0%)
  neighborhood_id               :  436 (100.0%)


## 10- 🆕 Data Modeling & Schema Proposal 
Based on data analysis showing 436 government offices with 19 columns following Table Structure Decision is made.

### Recommended Approach: Single Table Structure

Reasoning:

- Data Volume: 436 rows is a manageable size for a single table
- Query Simplicity: Most queries will likely search across all government offices regardless of type
- Unified Schema: All offices share the same core attributes (address, contact, location)
- Flexibility: Single table makes it easier to add new office types without schema changes
- Performance: With proper indexing, a single table will perform well for this volume

Alternative Considered: Multiple Tables by Office Type
❌ Not Recommended because:

- Low data volume doesn't justify the complexity
- Would require complex JOIN queries for common searches
- Harder to maintain consistency across tables

## 11- Proposed Table Structure
Table Name: government_offices_in_berlin

In [32]:
# to be written in SQL table government_offices_in_berlin
""""

CREATE TABLE government_offices_in_berlin (
    -- Primary Key
    office_id               BIGINT PRIMARY KEY,
    
    -- Foreign Key (References districts table)
    district_id             VARCHAR(10) NOT NULL,
    neighborhood_id         VARCHAR(10),
    
    -- Office Information
    office_name             VARCHAR(255),
    office_type             VARCHAR(100),
    
    -- Address Information
    street                  VARCHAR(255),
    housenumber             VARCHAR(20),
    postal_code             VARCHAR(10),
    city                    VARCHAR(100) DEFAULT 'Berlin',
    district                VARCHAR(100),
    neighborhood            VARCHAR(100),
    
    -- Contact Information
    phone_number            VARCHAR(50),
    email                   VARCHAR(255),
    website                 VARCHAR(500),
    
    -- Service Information
    opening_hours           TEXT,
    wheelchair_accessible   VARCHAR(20),
    
    -- Geospatial Information
    latitude                FLOAT,
    longitude               FLOAT,
    coordinate_type         VARCHAR(50),
    geometry                GEOMETRY(Point, 4326),
    
    -- Metadata
    created_at              TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at              TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    
    -- Indexes
    INDEX idx_district_id (district_id),
    INDEX idx_office_type (office_type),
    INDEX idx_postal_code (postal_code),
    INDEX idx_district (district),
    SPATIAL INDEX idx_geometry (geometry)
);

"""

'"\n\nCREATE TABLE government_offices_in_berlin (\n    -- Primary Key\n    office_id               BIGINT PRIMARY KEY,\n    \n    -- Foreign Key (References districts table)\n    district_id             VARCHAR(10) NOT NULL,\n    neighborhood_id         VARCHAR(10),\n    \n    -- Office Information\n    office_name             VARCHAR(255),\n    office_type             VARCHAR(100),\n    \n    -- Address Information\n    street                  VARCHAR(255),\n    housenumber             VARCHAR(20),\n    postal_code             VARCHAR(10),\n    city                    VARCHAR(100) DEFAULT \'Berlin\',\n    district                VARCHAR(100),\n    neighborhood            VARCHAR(100),\n    \n    -- Contact Information\n    phone_number            VARCHAR(50),\n    email                   VARCHAR(255),\n    website                 VARCHAR(500),\n    \n    -- Service Information\n    opening_hours           TEXT,\n    wheelchair_accessible   VARCHAR(20),\n    \n    -- Geospatial Informa

## 12- Key Design Decisions and column specifications

### 1. Primary Key
- `office_id` (OSM ID) serves as primary key
- Unique, stable identifier from OpenStreetMap

### 2. Foreign Keys
- `district_id` → References `berlin_districts` table
- Enables efficient joins for district-level queries

### 3. Data Types
- **VARCHAR** for text fields with reasonable length limits
- **TEXT** for `opening_hours` (variable length, complex format)
- **FLOAT** for coordinates (sufficient precision for Berlin)
- **GEOMETRY** for native spatial queries in PostGIS
- **TIMESTAMP** for audit trail


In [33]:
# Create schema documentation dataframe
schema_doc = pd.DataFrame({
    'Column Name': [
        'office_id', 'district_id', 'neighborhood_id', 'office_name', 'office_type',
        'street', 'housenumber', 'postal_code', 'city',
        'district', 'neighborhood', 'phone_number', 'email', 'website',
        'opening_hours', 'wheelchair_accessible', 'latitude', 'longitude',
        'coordinate_type', 'geometry', 'created_at', 'updated_at'
    ],
    'Data Type': [
        'BIGINT', 'VARCHAR(10)', 'VARCHAR(10)', 'VARCHAR(255)', 'VARCHAR(100)',
        'VARCHAR(255)', 'VARCHAR(20)', 'VARCHAR(10)', 'VARCHAR(100)',
        'VARCHAR(100)', 'VARCHAR(100)', 'VARCHAR(50)', 'VARCHAR(255)', 'VARCHAR(500)',
        'TEXT', 'VARCHAR(20)', 'FLOAT', 'FLOAT',
        'VARCHAR(50)', 'GEOMETRY', 'TIMESTAMP', 'TIMESTAMP'
    ],
    'Key': [
        'PRIMARY KEY', 'FOREIGN KEY', '', '', '',
        '', '', '', "DEFAULT 'Berlin'",
        '', '', '', '', '',
        '', '', '', '',
        '', '', 'DEFAULT NOW()', 'DEFAULT NOW()'
    ],
    'Description': [
        'Unique OSM ID', 'Links to districts table', 'Neighborhood identifier',
        'Official name of office', 'Type/category of office',
        'Street name', 'Building number', '5-digit postal code', 'City name',
        'District name', 'Neighborhood name', 'Contact phone', 'Contact email',
        'Official website URL', 'Service hours', 'Accessibility info',
        'Latitude (WGS84)', 'Longitude (WGS84)', 'Geometry type (Point/Polygon)',
        'PostGIS geometry object', 'Record creation time', 'Last update time'
    ],
    'Completeness': [
        '100%', '100%', '100%', '96.1%', '53.2%',
        '73.9%', '74.5%', '72.5%', '72.2%',
        '100%', '100%', '32.6%', '15.4%', '59.9%',
        '37.8%', '44.3%', '61.9%', '61.9%',
        '61.9%', '100%', 'Auto', 'Auto'
    ]
})

display(schema_doc)

,Column Name,Data Type,Key,Description,Completeness
0,office_id,BIGINT,PRIMARY KEY,Unique OSM ID,100%
1,district_id,VARCHAR(10),FOREIGN KEY,Links to districts table,100%
2,neighborhood_id,VARCHAR(10),,Neighborhood identifier,100%
3,office_name,VARCHAR(255),,Official name of office,96.1%
4,office_type,VARCHAR(100),,Type/category of office,53.2%
5,street,VARCHAR(255),,Street name,73.9%
6,housenumber,VARCHAR(20),,Building number,74.5%
7,postal_code,VARCHAR(10),,5-digit postal code,72.5%
8,city,VARCHAR(100),DEFAULT 'Berlin',City name,72.2%
9,district,VARCHAR(100),,District name,100%
